# 01 Stable Geometry Visualization

这个 notebook 只做 `01` 的稳定几何与解析链路可视化：读取几何审计结果、稳定 pilot/full H5 和解析评估结果，展示阈值热图、误差-几何关系、以及典型成功/失败样本。

In [ ]:
from pathlib import Path
import csv
import json
import sys

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
for parent in [ROOT] + list(ROOT.parents):
    if (parent / 'python_impl').exists() and (parent / 'README.md').exists():
        ROOT = parent
        break
SRC_DIR = ROOT / 'python_impl' / 'experiments' / 'hypothesis_validation' / '01_source_localization_anechoic_2d' / 'src'
if str(ROOT / 'python_impl') not in sys.path:
    sys.path.insert(0, str(ROOT / 'python_impl'))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from python_scripts.hypothesis_validation_common import estimate_source_position_from_tdoa, estimate_tdoa_from_gcc_triplet
from stable_geometry_pipeline import load_geometry_audit_bundle, load_stable_localization_summary

STEP_DIR = ROOT / 'python_impl' / 'experiments' / 'hypothesis_validation' / '01_source_localization_anechoic_2d'
AUDIT_DIR = STEP_DIR / 'results' / 'geometry_stability_audit'
H5_PATH = STEP_DIR / 'data' / 'source_localization_anechoic_2d_l1_stable.h5'
RESULT_DIR = STEP_DIR / 'results' / 'L1_stable'
SAMPLE_SPLIT = 'iid_test'
print('ROOT =', ROOT)
print('AUDIT_DIR =', AUDIT_DIR)
print('H5_PATH =', H5_PATH)
print('RESULT_DIR =', RESULT_DIR)

In [ ]:
audit = load_geometry_audit_bundle(AUDIT_DIR)
audit_summary = audit['summary']
threshold_df = pd.DataFrame(audit['threshold_grid'])
sample_df = pd.DataFrame(audit['sample_metrics'])
threshold_df = threshold_df.apply(pd.to_numeric, errors='ignore')
sample_df = sample_df.apply(pd.to_numeric, errors='ignore')
display(pd.Series(audit_summary['selected_thresholds'], name='selected_thresholds'))
display(threshold_df.sort_values(['retention_overall', 'gcc_phat_iid_test_p90_m'], ascending=[False, True]).head(10))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
axes[0].scatter(sample_df['triangle_area'], sample_df['gcc_phat_position_error_m'], s=5, alpha=0.12)
axes[0].set_xlabel('triangle_area')
axes[0].set_ylabel('gcc_phat_position_error_m')
axes[0].set_title('Error vs Triangle Area')
axes[0].grid(True, alpha=0.25)

inside = sample_df['inside_convex_hull'].to_numpy() > 0.5
axes[1].scatter(sample_df.loc[~inside, 'jacobian_condition'], sample_df.loc[~inside, 'gcc_phat_position_error_m'], s=5, alpha=0.10, label='outside')
axes[1].scatter(sample_df.loc[inside, 'jacobian_condition'], sample_df.loc[inside, 'gcc_phat_position_error_m'], s=5, alpha=0.18, label='inside')
axes[1].set_xscale('log')
axes[1].set_xlabel('jacobian_condition')
axes[1].set_ylabel('gcc_phat_position_error_m')
axes[1].set_title('Error vs Condition Number')
axes[1].legend()
axes[1].grid(True, alpha=0.25)

pivot = threshold_df.pivot(index='area_min', columns='cond_max', values='retention_overall')
im = axes[2].imshow(pivot.to_numpy(), origin='lower', aspect='auto')
axes[2].set_xticks(np.arange(pivot.shape[1]), labels=[str(int(v)) for v in pivot.columns])
axes[2].set_yticks(np.arange(pivot.shape[0]), labels=[f'{v:.2f}' for v in pivot.index])
axes[2].set_xlabel('cond_max')
axes[2].set_ylabel('area_min')
axes[2].set_title('Retention Heatmap')
fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
plt.show()

In [ ]:
stable_summary = load_stable_localization_summary(RESULT_DIR)
rows = []
for label in ['analytic_plain_gcc', 'analytic_gcc_phat']:
    for split in ['iid_test', 'geom_test']:
        metrics = stable_summary[label][split]
        rows.append({
            'method': label,
            'split': split,
            'median_m': metrics['median_m'],
            'p90_m': metrics['p90_m'],
            'mean_m': metrics['mean_m'],
            'max_m': metrics['max_m'],
        })
display(pd.DataFrame(rows))

with h5py.File(H5_PATH, 'r') as h5:
    ref_positions = np.asarray(h5['raw/ref_positions'], dtype=np.float32)
    source_position = np.asarray(h5['raw/source_position'], dtype=np.float32)
    gcc_phat = np.asarray(h5['raw/gcc_phat'], dtype=np.float32)
    true_tdoa = np.asarray(h5['raw/true_tdoa'], dtype=np.float32)
    geom_metrics = np.asarray(h5['raw/geometry_metrics'], dtype=np.float32)
    cfg = json.loads(h5.attrs['config_json'])
display(pd.DataFrame(geom_metrics, columns=json.loads(h5.attrs['geometry_metric_names_json'])).describe().T[['mean', 'std', 'min', 'max']].head(12))

In [ ]:
analytic_rows = []
with (RESULT_DIR / 'analytic_sample_errors.csv').open('r', encoding='utf-8', newline='') as handle:
    analytic_rows = list(csv.DictReader(handle))
analytic_df = pd.DataFrame(analytic_rows)
analytic_df['sample_index'] = analytic_df['sample_index'].astype(int)
analytic_df['position_error_m'] = analytic_df['position_error_m'].astype(float)
sub = analytic_df[(analytic_df['method'] == 'gcc_phat') & (analytic_df['split'] == SAMPLE_SPLIT)].sort_values('position_error_m')
success_idx = int(sub.iloc[0]['sample_index'])
failure_idx = int(sub.iloc[-1]['sample_index'])

def plot_sample(sample_index: int, title: str):
    gcc_pred = estimate_source_position_from_tdoa(
        estimate_tdoa_from_gcc_triplet(gcc_phat[sample_index]),
        ref_positions[sample_index],
        tuple(cfg['plane_room_size']),
        int(cfg['fs']),
        float(cfg['c']),
    )
    oracle_pred = estimate_source_position_from_tdoa(
        true_tdoa[sample_index],
        ref_positions[sample_index],
        tuple(cfg['plane_room_size']),
        int(cfg['fs']),
        float(cfg['c']),
    )
    fig, ax = plt.subplots(figsize=(5.5, 5.5), constrained_layout=True)
    tri = ref_positions[sample_index]
    ax.scatter(tri[:, 0], tri[:, 1], s=90, color='tab:green', label='refs')
    ax.scatter(source_position[sample_index, 0], source_position[sample_index, 1], s=100, color='tab:blue', label='true')
    ax.scatter(gcc_pred[0], gcc_pred[1], s=100, color='tab:orange', label='gcc_phat')
    ax.scatter(oracle_pred[0], oracle_pred[1], s=90, color='tab:red', label='oracle', marker='x')
    ax.set_xlim(0, cfg['plane_room_size'][0])
    ax.set_ylim(0, cfg['plane_room_size'][1])
    ax.grid(True, alpha=0.25)
    ax.legend()
    area = geom_metrics[sample_index, 0]
    cond = geom_metrics[sample_index, 10]
    ax.set_title(f"{title} | idx={sample_index} | area={area:.3f} | cond={cond:.2f}")
    plt.show()

plot_sample(success_idx, f'{SAMPLE_SPLIT} success')
plot_sample(failure_idx, f'{SAMPLE_SPLIT} failure')